# Step 4: Claim extraction (transcript → checkable claims)
A transcript is mostly filler, opinion and chatter. Only a few sentences are **checkable factual claims**. This notebook trains the part that finds them.

**What it does**
1. Fine-tunes DeBERTa on **CheckThat! 2024 (English)**: sentences labeled *check-worthy* (Yes) or not (No). It has official train / validation / test splits.
2. Picks the decision threshold on **validation only**, then scores the official **test** split once.
3. Adds a transcript splitter (handles timestamps and unpunctuated auto-captions) and an `extract_claims()` function the demo will call.
4. Saves the model to Drive as `claim_detector/`.

**Honest limit:** the training sentences come from political debates, not health videos. Expect weaker results on casual health talk. That is checked on real transcripts in the real-claims test set.

**Runtime → T4 GPU.** Approve the Drive pop-up. About 8 minutes.

In [ ]:
%pip install -q "datasets==3.6.0" "transformers>=4.46,<5" accelerate sentencepiece protobuf scikit-learn pandas

In [ ]:
import os, re, json, numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, set_seed)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/veritas"
except ImportError:                       # not on Colab (local test only)
    DRIVE = "veritas_local"
os.makedirs(DRIVE, exist_ok=True)
SAVE_DIR = f"{DRIVE}/claim_detector"
CKPT = "/content/ckpt_claims" if os.path.isdir("/content") else "ckpt_claims"   # local disk is fine: training is short
BASE = "microsoft/deberta-v3-base"
MAXLEN, EPOCHS, LR, BS = 128, 3, 2e-5, 32
set_seed(42)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## 1. Data
`class_label`: **Yes** = a checkable factual claim, **No** = anything else (opinion, question, filler). About 24% of training sentences are Yes.

In [ ]:
ds = load_dataset("iai-group/clef2024_checkthat_task1_en")
def prep(split):
    d = pd.DataFrame(ds[split])
    return pd.DataFrame(dict(text=d["Text"].astype(str).str.strip(), label=(d["class_label"] == "Yes").astype(int)))
TR, VA, TE = prep("train"), prep("validation"), prep("test")
for n, d in [("train", TR), ("validation", VA), ("test", TE)]:
    print(f"{n:11s} {len(d):6d} sentences | Yes: {d.label.mean():.1%}")
print("\nexamples of Yes:"); [print("  -", t) for t in TR[TR.label == 1].text.sample(4, random_state=1)]
print("examples of No:");  [print("  -", t) for t in TR[TR.label == 0].text.sample(4, random_state=1)]
print("\nsentence length (words), median:", int(TR.text.str.split().str.len().median()))

## 2. Train
Standard fine-tune, 3 epochs. The best epoch is picked by **F1 on the Yes class** (the CheckThat! official metric), since accuracy is misleading with 76% No.

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
def enc(d):
    return Dataset.from_dict(dict(text=d.text.tolist(), label=d.label.tolist())).map(
        lambda b: tok(b["text"], truncation=True, max_length=MAXLEN), batched=True, remove_columns=["text"])
dtr, dva, dte = enc(TR), enc(VA), enc(TE)

def metrics(p):
    pred = p.predictions.argmax(-1)
    return dict(f1=f1_score(p.label_ids, pred), precision=precision_score(p.label_ids, pred), recall=recall_score(p.label_ids, pred))

model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=2)
args = TrainingArguments(output_dir=CKPT, num_train_epochs=EPOCHS, learning_rate=LR, per_device_train_batch_size=BS,
                         per_device_eval_batch_size=64, warmup_ratio=0.1, weight_decay=0.01,
                         eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
                         metric_for_best_model="f1", fp16=torch.cuda.is_available(), report_to="none", seed=42)
trainer = Trainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva, data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=metrics)
trainer.train()

## 3. Pick the threshold on validation, score test once
The model outputs P(check-worthy). The cut-off is a choice: lower = catches more claims (more false alarms), higher = stricter. We pick the value that maximizes F1 on **validation**, then report **test** at that value.

In [ ]:
def probs(dset):
    o = trainer.predict(dset)
    z = o.predictions - o.predictions.max(-1, keepdims=True)
    return (np.exp(z) / np.exp(z).sum(-1, keepdims=True))[:, 1], o.label_ids
pv, yv = probs(dva); pt, yt = probs(dte)
assert (yv == VA.label.values).all() and (yt == TE.label.values).all()

grid = np.round(np.arange(0.1, 0.91, 0.05), 2)
f1s = [f1_score(yv, pv >= t) for t in grid]
THR = float(grid[int(np.argmax(f1s))])
print(pd.DataFrame({"threshold": grid, "val_F1": np.round(f1s, 3)}).to_string(index=False))
print("\nchosen threshold (validation):", THR)

rows = []
for name, p, y in [("validation", pv, yv), ("test (scored once)", pt, yt)]:
    for lab, t in [("thr=0.5", 0.5), (f"thr={THR}", THR)]:
        rows.append(dict(split=name, cut=lab, F1_yes=round(f1_score(y, p >= t), 3),
                         precision=round(precision_score(y, p >= t), 3), recall=round(recall_score(y, p >= t), 3)))
print(); print(pd.DataFrame(rows).to_string(index=False))
print("\nReference: the strongest CheckThat! 2024 English systems scored about 0.80-0.90 F1 on the Yes class.")

In [ ]:
d = VA.assign(p=pv)
print("Most confident FALSE ALARMS on validation (model says claim, label says no):")
for _, r in d[(d.label == 0)].sort_values("p", ascending=False).head(6).iterrows(): print(f"  {r.p:.2f}  {r.text[:130]}")
print("\nMost confident MISSES on validation (label says claim, model says no):")
for _, r in d[(d.label == 1)].sort_values("p").head(6).iterrows(): print(f"  {r.p:.2f}  {r.text[:130]}")

## 4. Save to Drive

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR); tok.save_pretrained(SAVE_DIR)
json.dump(dict(base=BASE, threshold=THR, max_len=MAXLEN, data="iai-group/clef2024_checkthat_task1_en"),
          open(f"{SAVE_DIR}/extract_config.json", "w"))
print(sorted(os.listdir(SAVE_DIR)))

## 5. From a transcript to claims
Real transcripts are messy: timestamps, no punctuation (auto-captions), run-on speech. The splitter:
1. strips timestamps like `[00:12]` or `0:12`,
2. splits on `. ! ?`,
3. cuts any remaining piece longer than 40 words into 25-word windows (the auto-caption case).

`extract_claims()` then keeps sentences scoring at or above the threshold, and returns every sentence's score so the UI can show why something was or wasn't picked.

In [ ]:
def split_transcript(text, max_words=40, window=25):
    text = re.sub(r"\[?\(?\b\d{1,2}:\d{2}(?::\d{2})?\b\]?\)?", " ", text)      # timestamps
    text = re.sub(r"\s+", " ", text).strip()
    out = []
    for s in re.split(r"(?<=[.!?])\s+", text):
        w = s.split()
        if len(w) <= max_words:
            if len(w) >= 4: out.append(s)
        else:
            out += [" ".join(w[i:i + window]) for i in range(0, len(w), window) if len(w[i:i + window]) >= 4]
    return out

@torch.no_grad()
def claim_probs(sents, bs=64):
    model.eval(); res = []
    for i in range(0, len(sents), bs):
        b = tok(sents[i:i + bs], truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt").to(model.device)
        res.append(torch.softmax(model(**b).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(res) if res else np.array([])

def extract_claims(transcript, thr=None):
    thr = THR if thr is None else thr
    sents = split_transcript(transcript); p = claim_probs(sents)
    return [dict(sentence=s, p_claim=float(x), is_claim=bool(x >= thr)) for s, x in zip(sents, p)]

In [ ]:
# An invented demo transcript (not real data): claims, opinions, filler and an off-topic fact mixed together.
DEMO = '''[0:01] Hey everyone, welcome back to the channel. Today I want to talk about sleep and supplements.
[0:12] So first, I take magnesium glycinate every night and honestly it cures my insomnia. Studies show that vitamin D
supplementation reduces the risk of respiratory infections. I personally think cold plunges are amazing.
[0:40] Masks do not reduce the spread of COVID-19, that's just a fact. Intermittent fasting is the best diet in my opinion.
Drinking lemon water every morning cures cancer. Make sure you like and subscribe. The Eiffel Tower is 330 meters tall.
Statins cause more harm than good for most people. Anyway, let me know what you think in the comments.'''
r = pd.DataFrame(extract_claims(DEMO))
pd.set_option("display.max_colwidth", 95)
print(r.assign(p_claim=r.p_claim.round(2)).to_string(index=False))

nopunct = " ".join(DEMO.split()).lower().replace(".", "").replace(",", "")        # auto-caption style: no punctuation
print("\nUnpunctuated auto-caption version ->", len(split_transcript(nopunct)), "chunks;",
      sum(x["is_claim"] for x in extract_claims(nopunct)), "flagged")

## What to check before Step 5/6
1. **Test F1 (Yes class)** at the chosen threshold. Above about 0.75 is good enough for the demo. The model only needs to *surface* candidate claims, since the verdict system and its "no evidence" answer filter the rest.
2. **Recall vs precision.** For a fact-checker, missing a claim (low recall) is worse than a false alarm (low precision). A false alarm just gets "no evidence found". If recall is low, lower the threshold.
3. **The demo transcript.** The health claims (magnesium, vitamin D, masks, lemon water, statins) should score high. "Welcome back", "like and subscribe" and opinions should score low. The Eiffel Tower fact may score high, which is fine: it will be filtered later as not health-related.
4. **The unpunctuated version** should still give a similar number of flagged claims.